In [2]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import sys
from tqdm import tqdm
from scipy.fft import fft2

sys.path.append('..')

# Import your models
from models.PRUSR.model import PRUSR
from models.SRCNN.model import SRCNN
from models.DSCMS.model import DSCMS

# Import your datasets and metrics
from datasets.super_res_dataset import SuperResDataset
from metrics.image_metrics import mseLoss, ssimLoss, psnrLoss, lpipsLoss, fsimLoss, epiLoss
from metrics.physical_metrics import LossTKE
from metrics.kinetic_energy_spectrum import KineticEnergySpectrumLoss, WeightedKineticEnergySpectrumLoss
from typing import Tuple

In [5]:
# Initialize your evaluation dataset based on your configuration[cite: 5]
test_dataset = SuperResDataset(
    hr_files=[
        "../hycom_numpy/hycom_2022.npy",
        "../hycom_numpy/hycom_2023.npy",
        "../hycom_numpy/hycom_2024.npy"
    ],
    downsample_factor=1,
    freq_threshold=10
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,  
    shuffle=False  
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Loading datasets...
../hycom_numpy/hycom_2022.npy: (365, 128, 128, 2)
../hycom_numpy/hycom_2023.npy: (365, 128, 128, 2)
../hycom_numpy/hycom_2024.npy: (244, 128, 128, 2)
Loaded 974 samples.
Dataset shape: (974, 128, 128, 2)
Using device: cuda


In [6]:
def evaluate_model(model, test_loader, device):
    """Evaluates the model across multiple metrics for u and v channels.[cite: 5]"""
    model.eval()
    
    metrics = {
        'mse': {'u': [], 'v': []},
        'ssim': {'u': [], 'v': []},
        'psnr': {'u': [], 'v': []},
        'lpips': {'u': [], 'v': []},
        'fsim': {'u': [], 'v': []},
        'epi':{'u': [], 'v': []},
        'tke': {'u': [], 'v': []}
    }
    
    with torch.no_grad():
        for idx, (lr, hr) in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
            lr, hr = lr.to(device), hr.to(device)
            output = model(lr)
            # Compute metrics for each channel[cite: 5]
            for channel in range(2):  
                channel_label = 'u' if channel == 0 else 'v'
                metrics['mse'][f'{channel_label}'].append(mseLoss(output, hr, channel, idx))
                metrics['ssim'][f'{channel_label}'].append(ssimLoss(output, hr, channel, idx))
                metrics['psnr'][f'{channel_label}'].append(psnrLoss(output, hr, channel, idx))
                metrics['lpips'][f'{channel_label}'].append(lpipsLoss(output, hr, channel, idx))
                metrics['fsim'][f'{channel_label}'].append(fsimLoss(output, hr, channel, idx))
                metrics['epi'][f'{channel_label}'].append(epiLoss(output, hr, channel, idx))
                metrics['tke'][f'{channel_label}'].append(LossTKE(output, hr, channel, idx))

    # Aggregate results[cite: 5]
    results = {k: {'u': {'mean': np.mean(v['u']), 'std': np.std(v['u'])},
                    'v': {'mean': np.mean(v['v']), 'std': np.std(v['v'])}}
               for k, v in metrics.items()}
    return results

def interpolate_baseline(lr, mode, downsample_factor=4):
    """Generate baseline output by interpolating the coarse field back to HR size.[cite: 5]"""
    coarse_h = lr.shape[2] // downsample_factor
    coarse_w = lr.shape[3] // downsample_factor
    coarse = F.adaptive_avg_pool2d(lr, (coarse_h, coarse_w))
    output = F.interpolate(coarse, size=(lr.shape[2], lr.shape[3]), mode=mode,
                           align_corners=False if mode != 'nearest' else None)
    return output

In [9]:
from collections import OrderedDict

def get_model_instance(model_name):
    """Helper to initialize the correct architecture."""
    if model_name == 'PRUSR':
        return PRUSR(2, 2)
    elif model_name == 'SRCNN':
        return SRCNN(2, 2)
    elif model_name == 'DSCMS':
        return DSCMS(2, 2)
    else:
        raise ValueError(f"Unknown model: {model_name}")

models_to_test = ['SRCNN', 'DSCMS', 'PRUSR']
base_model_dir = "/home/lcadame/Vinicius_Thesis/Vinicius/models" 

all_evaluation_results = {}

for model_name in models_to_test:
    search_pattern = os.path.join(base_model_dir, model_name, "*", "output", "weights", f"{model_name}_best.pth")
    matching_paths = glob.glob(search_pattern)
    
    if not matching_paths:
        print(f"Skipping {model_name}: No matching weights found in {base_model_dir}/{model_name}/*/")
        continue
        
    for weight_path in matching_paths:
        strategy_folder_name = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(weight_path))))
        folder_lower = strategy_folder_name.lower()
        
        if 'ke' in folder_lower:
            strategy_category = 'mse_lpips_ke'
        elif 'lpips' in folder_lower:
            strategy_category = 'mse_lpips'
        elif 'mse' in folder_lower:
            strategy_category = 'mse'
        else:
            print(f"Skipping {strategy_folder_name}: Does not match expected strategy keywords.")
            continue
            
        experiment_name = f"{model_name}_{strategy_folder_name}"
        
        print(f"\n--- Evaluating {experiment_name} ---")
        print(f"Loading weights from: {weight_path}")
        
        # 1. Initialize BASE model architecture (Do NOT wrap in DataParallel yet)
        model = get_model_instance(model_name)
        
        # 2. Load checkpoint
        checkpoint = torch.load(weight_path, map_location=torch.device(device))
        
        # Extract the state dict
        state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
        
        # 3. Clean the state_dict keys (removes 'module.' if it was saved with DataParallel)
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k.replace('module.', '') if k.startswith('module.') else k
            new_state_dict[name] = v
            
        # 4. Load the cleaned weights into the base model
        model.load_state_dict(new_state_dict)
        
        # 5. Move to device and wrap in DataParallel AFTER loading
        model = model.to(device)
        model = torch.nn.DataParallel(model)
            
        # 6. Evaluate Model
        results = evaluate_model(model, test_loader, device)
        all_evaluation_results[experiment_name] = results


--- Evaluating SRCNN_test_composite_loss_mse_lpips_0.6_0.4_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_lpips_0.6_0.4_05_08_2026/output/weights/SRCNN_best.pth



--- Evaluating SRCNN_test_composite_loss_mse_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_05_08_2026/output/weights/SRCNN_best.pth



--- Evaluating SRCNN_test_composite_loss_mse_lpips_ke_0.6_0.2_0.2_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_lpips_ke_0.6_0.2_0.2_05_08_2026/output/weights/SRCNN_best.pth



--- Evaluating DSCMS_test_composite_loss_mse_lpips_0.6_0.4_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/DSCMS/test_composite_loss_mse_lpips_0.6_0.4_05_08_2026/output/weights/DSCMS_best.pth



--- Evaluating DSCMS_test_composite_loss_mse_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/DSCMS/test_composite_loss_mse_05_08_2026/output/weights/DSCMS_best.pth



--- Evaluating DSCMS_test_composite_loss_mse_lpips_ke_0.6_0.2_0.2_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/DSCMS/test_composite_loss_mse_lpips_ke_0.6_0.2_0.2_05_08_2026/output/weights/DSCMS_best.pth


Skipping factory_run_test: Does not match expected strategy keywords.

--- Evaluating PRUSR_test_composite_loss_mse_lpips_0.6_0.4_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/PRUSR/test_composite_loss_mse_lpips_0.6_0.4_05_08_2026/output/weights/PRUSR_best.pth



--- Evaluating PRUSR_test_composite_loss_mse_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/PRUSR/test_composite_loss_mse_05_08_2026/output/weights/PRUSR_best.pth



--- Evaluating PRUSR_test_composite_loss_mse_lpips_ke_0.6_0.2_0.2_05_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/PRUSR/test_composite_loss_mse_lpips_ke_0.6_0.2_0.2_05_08_2026/output/weights/PRUSR_best.pth



--- Evaluating PRUSR_test_composite_loss_mse_10_08_2026 ---
Loading weights from: /home/lcadame/Vinicius_Thesis/Vinicius/models/PRUSR/test_composite_loss_mse_10_08_2026/output/weights/PRUSR_best.pth


In [10]:
# Flatten the nested dictionary results into a pandas DataFrame for readable analysis
rows = []
for experiment, metrics in all_evaluation_results.items():
    row_data = {'Experiment': experiment}
    for metric_name, channels in metrics.items():
        row_data[f'{metric_name}_u_mean'] = channels['u']['mean']
        row_data[f'{metric_name}_v_mean'] = channels['v']['mean']
    rows.append(row_data)

df_results = pd.DataFrame(rows)
df_results.to_csv("evaluation_results_summary.csv", index=False)

print("\n--- Final Evaluation Summary ---")
display(df_results)


--- Final Evaluation Summary ---


,Experiment,mse_u_mean,mse_v_mean,ssim_u_mean,ssim_v_mean,psnr_u_mean,psnr_v_mean,lpips_u_mean,lpips_v_mean,fsim_u_mean,fsim_v_mean,epi_u_mean,epi_v_mean,tke_u_mean,tke_v_mean
0,SRCNN_test_composite_loss_mse_lpips_0.6_0.4_05...,0.000550,0.000591,0.975624,0.977651,40.981606,41.488785,0.074015,0.070619,0.741628,0.791143,0.030220,0.026680,0.009158,0.009994
1,SRCNN_test_composite_loss_mse_05_08_2026,0.001156,0.001184,0.951723,0.955651,37.750027,38.495441,0.230478,0.228021,0.635549,0.700904,0.035203,0.030940,0.022276,0.023915
2,SRCNN_test_composite_loss_mse_lpips_ke_0.6_0.2...,0.000629,0.000672,0.972783,0.975331,40.436653,40.985809,0.080616,0.076829,0.725684,0.776789,0.028702,0.025384,0.011353,0.012244
3,DSCMS_test_composite_loss_mse_lpips_0.6_0.4_05...,0.000293,0.000362,0.987279,0.988528,43.732822,43.603462,0.038872,0.038245,0.824362,0.863869,0.029773,0.025836,0.003927,0.004476
4,DSCMS_test_composite_loss_mse_05_08_2026,0.000960,0.001142,0.957280,0.957031,38.554043,38.614548,0.203577,0.208403,0.645376,0.712846,0.034130,0.030143,0.019377,0.022431
5,DSCMS_test_composite_loss_mse_lpips_ke_0.6_0.2...,0.000462,0.000514,0.981117,0.984076,41.766624,42.084782,0.057661,0.053255,0.774938,0.830384,0.028980,0.025535,0.007220,0.007369
6,PRUSR_test_composite_loss_mse_lpips_0.6_0.4_05...,0.000375,0.000345,0.980670,0.985192,42.613949,43.819912,0.043717,0.037622,0.797622,0.860260,0.029649,0.026301,0.004924,0.004319
7,PRUSR_test_composite_loss_mse_05_08_2026,0.000773,0.000845,0.962391,0.965192,39.522541,39.958012,0.186369,0.181895,0.666071,0.739457,0.032827,0.028638,0.016391,0.017590
8,PRUSR_test_composite_loss_mse_lpips_ke_0.6_0.2...,0.000448,0.000466,0.978791,0.981102,41.888836,42.545937,0.055197,0.051911,0.765055,0.821736,0.028933,0.025546,0.007009,0.007244
9,PRUSR_test_composite_loss_mse_10_08_2026,0.000793,0.000858,0.961774,0.964787,39.406353,39.890381,0.187718,0.183833,0.663348,0.737678,0.033040,0.028835,0.016673,0.017819
